**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/elie-2019"], check=True)
    os.chdir("/content/encore/entries/elie-2019")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Elie & Theunissen (2019) on CRCNS aa-4

Paper: Elie JE, Theunissen FE (2019). Invariant neural responses for sensory categories revealed by the
time-varying information for communication calls. PLoS Comput Biol 15(9):e1006698,
doi:10.1371/journal.pcbi.1006698.

Dataset: CRCNS aa-4, "Simultaneous extracellular recordings of avian auditory neurons in zebra finches presented
with all the repertoire of vocalizations used by this species for vocal communication", doi:10.6080/K00C4T06,
data description version 0.6 (July 15, 2019). CRCNS data sets are not versioned, so the version is recorded as
`unversioned`; the only published state is the one in `aa-4/live/`, and it is the one read here. The files are
streamed from the public S3 bucket `crcnsarchive`; their MD5 sums in `aa-4/live/checksums.md5`, read on
2026-11-13, are:

| file | md5 |
|---|---|
| data/BlaBro09xxF.tar.gz | ee387cc051e5fa5ee66a5f4db5d8bae6 |
| data/GreBlu9508M.tar.gz | 16f1bb223946288f8b426d6382f06510 |
| data/LblBlu2028M.tar.gz | 80c181eb8bd70ddbbd9b3aa46587dfb5 |
| data/WhiBlu5396M.tar.gz | 2ba7776cd4130d0090b13e5f5c0359fd |
| data/WhiWhi4522M.tar.gz | c47d6660019a2a0d1e3375ce7da68bf1 |
| data/YelBlu6903F.tar.gz | 6027933cdf765dd0b67f811b1a133095 |

Outside input: the tutorial model cell `ModelCell_InfoCumInfo_Tutorial.mat` of the paper's code repository,
github.com/julieelie/PoissonTimeVaryingInfo at commit 978020328321c7ed83be12c0152bed85ec77e218 (claim
fig1-model-neurons, `data: elsewhere`).

Claims file: `claims/elie-2019.yaml`, written from the version of record (PMC6762074) with its S1 to S7 Fig
legends.

| claim | status here |
|---|---|
| units-404 | blocked: the 914 single auditory units and the 404 selected ones cannot be identified in the files |
| units-337, s1-temporal-resolution, fig2a-poisson-test, fig2cd-fano-factor, fig2b-noise-correlations, s2-spike-counts | blocked: rest on the 404 units (variants on all auditory single units shown, not deciding) |
| fig6a-onset-sustained-information, fig6c-new-information, fig6b-fixed-rate-fraction, fig7a-time-constants, fig7b-k300, k300-rate-regression, fig9a-mean-cii, fig9a-top-quartile, s7-invariant-information, fig9b-cii-tau-ratio, fig9c-cii-k300, cii-unimodal | blocked: rest on the 337 units of units-337 |
| fig3-5-example-neurons | blocked: the paper does not name the three units, and the S4 to S6 Fig images were not available |
| table1-anatomy-counts, fig10-cii-distance, fig10-cii-area-anova, fig10-tau-k300-distance, fig10-no-anatomy-categorical | blocked: no anatomical location in the data set (file attributes, `class_info` and `extra_info` listed below) |
| fig1-model-neurons | tested, on the one model neuron the paper's code repository holds |

## Loading

Each bird's tar.gz is streamed and its h5 files are read in memory (`elie.load_all`); only spike times from -600 to 600 ms around stimulus onset, the stimulus attributes, the stored coherence and the paths of the `class_info` and `extra_info` datasets are kept. The spike snippet files (`.f32`) are read as they pass in the stream and give each spike-sorted unit its snippet SNR (`elie.stream_bird`); the snippets are then dropped.

In [ ]:
import time, resource, collections
import numpy as np, pandas as pd
from scipy import stats
import elie

t0 = time.time()
U = elie.load_all()
print(f"loaded {len(U)} h5 files in {time.time() - t0:.0f} s; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB (this process)")

## Data check

Counts of files, units, stimuli and trials, and the ranges of the variables used, against the data description and the paper.

In [ ]:
pd.set_option("display.width", 200)
df = pd.DataFrame([{
    "bird": u["bird"], "file": u["file"], "site": u.get("site"), "electrode": u.get("electrode"),
    "sortType": u.get("sortType"), "ldepth": u.get("ldepth"), "rdepth": u.get("rdepth"),
    "spike_sorted_name": "_ss" in u["file"], "groups": "+".join(u["groups"]),
    "iar": elie.iar(u), "n_call_stims": len(elie.call_stims(u)) if elie.is_single(u) else np.nan,
    "n_stims_all": len(u["stims"]),
    "n_stims_call_protocol": sum(s["group"].startswith("Call") for s in u["stims"]), "n_unsorted_trials": u["n_unsorted_trials"],
} for u in U])
df["single"] = [elie.is_single(u) for u in U]
print("h5 files by bird and sortType:")
print(pd.crosstab(df.bird, df.sortType.astype(str), margins=True))
print("\nfiles named as spike sorted (_ss) by sortType:")
print(pd.crosstab(df.spike_sorted_name, df.sortType.astype(str)))
print("\nsites per bird:", df.groupby("bird").site.nunique().to_dict())
print("depths (um), left:", np.nanmin(df.ldepth.astype(float)), "to", np.nanmax(df.ldepth.astype(float)),
      "; right:", np.nanmin(df.rdepth.astype(float)), "to", np.nanmax(df.rdepth.astype(float)))
S = [u for u in U if elie.is_single(u)]
sd = df[df.single]
print(f"\nsingle units: {len(S)}; with I_AR >= 2.3 bits/s: {(sd.iar >= 2.3).sum()}; I_AR range "
      f"{sd.iar.min():.2f} to {sd.iar.max():.2f} bits/s")
print(f"call stimuli per single unit (call protocol): mean {sd.n_call_stims.mean():.1f}, sd {sd.n_call_stims.std():.1f}, "
      f"min {sd.n_call_stims.min():.0f}, max {sd.n_call_stims.max():.0f}")
print(f"all stimuli per single unit (any type, every group): mean {sd.n_stims_all.mean():.1f}, sd {sd.n_stims_all.std():.1f}, "
      f"min {sd.n_stims_all.min()}, max {sd.n_stims_all.max()}")
print(f"all stimuli per single unit (any type, call protocol): mean {sd.n_stims_call_protocol.mean():.1f}, "
      f"sd {sd.n_stims_call_protocol.std():.1f}, min {sd.n_stims_call_protocol.min()}, max {sd.n_stims_call_protocol.max()}")
print("call and song stimuli kept (call types of selVoc.m) per protocol group, over single units:")
grp = pd.DataFrame([{"group": s["group"], "file": u["file"], "bird": u["bird"]} for u in S for s in elie.call_stims(u, "")])
print(grp.groupby("group").agg(units=("file", "nunique"), unit_stimuli=("file", "size")).to_string())
dup_in = dup_across = 0
for u in S:
    md5 = collections.defaultdict(set)
    n = collections.Counter()
    for s in elie.call_stims(u, ""):
        md5[s["stim_md5"]].add(s["group"]); n[(s["group"], s["stim_md5"])] += 1
    dup_across += sum(len(v) > 1 for v in md5.values())
    dup_in += sum(c > 1 for c in n.values())
print(f"stim_md5 present in more than one group of the same unit: {dup_across}; repeated within a group: {dup_in}")
ntr = collections.Counter(len(s["trials"]) for u in S for s in elie.call_stims(u))
print("trials per call stimulus (call protocol):", dict(sorted(ntr.items())))
ntr_all = collections.Counter(len(s["trials"]) for u in S for s in elie.call_stims(u, ""))
print("trials per call stimulus (every group):", dict(sorted(ntr_all.items())))
print("call protocol groups per single unit:", dict(collections.Counter(
    "+".join(sorted({s["group"] for s in elie.call_stims(u)})) for u in S)))
print("call stimuli by call type (unit x stimulus):",
      dict(collections.Counter(s["ct"] for u in S for s in elie.call_stims(u))))
allsp = np.concatenate([np.concatenate(s["trials"]) for u in S for s in elie.call_stims(u) if sum(map(len, s["trials"]))])
print(f"spike times kept: {allsp.size} spikes, from {allsp.min():.4f} to {allsp.max():.4f} s around onset (unit s)")
dur = np.array([s["stim_duration"] for u in S for s in elie.call_stims(u)], float)
print(f"stim_duration of call stimuli: {dur.min():.3f} to {dur.max():.3f} s, median {np.median(dur):.3f}")
print("trials with stored spike times out of order:", int(df.n_unsorted_trials.sum()))

Checks against the description and the paper, from the output above:

- The data description says 688 multi-units were recorded and 914 single auditory units were saved, one h5 file
  each. The files hold 845 unsorted (`tdt`) multi-unit files and 1453 spike-sorted (`_ss`) files, of which 1281
  are labelled single units (`sortType` "single", or "singl" in three files), 63 multi and 109 noise. The 914
  are not marked. This is recorded as a finding.
- Six birds, as described; WhiBlu5396M is the bird missing from the paper's Table 1. The description's file
  organisation section names it WhiBlu5896M once.
- Stimuli. Each unit's file holds the call protocol (one group, `Call`, `Call1`, `Call1c`, `Call2` or `Call3`)
  and, for some units, other protocols (`Sel1` to `Sel7`, `Mask1`, `STRF1`) that also contain call and song
  stimuli; no stimulus (`stim_md5`) appears in two groups of a unit. Call stimuli are taken from the call protocol
  only (`elie` docstring). All stimuli of the call protocol: mean 115.3 per single unit (sd 34.2, 34 to 133),
  against the paper's 114 (sd 22, 34 to 123): the mean and the minimum agree, the spread and the maximum do not.
  All stimuli of every group: mean 121.9 (73 to 137), against the description's 144 (64 to 153), which matches
  neither count. Call stimuli of the nine call types in the call protocol: mean 90.7 (34 to 110).
- Trials per call stimulus in the call protocol: 9 to 12, with 1874 unit-stimulus pairs at 12, against
  "min=9, max=11".
- Spike times are in seconds from stimulus onset (range -0.6 to 0.6 s as kept); no trial holds spike times out of
  order. `stim_duration` (0.08 to 2.9 s, median 2.5 s) is longer than most calls; it is not used.


No recording is deposited twice: the spike trains of every single unit to its call stimuli are compared.

In [ ]:
import hashlib
keys = collections.defaultdict(list)
for u in S:
    st = sorted(elie.call_stims(u), key=lambda s: s["stim_md5"])
    h = hashlib.md5()
    for s in st:
        for t in s["trials"]:
            h.update(np.asarray(t, np.float32).tobytes()); h.update(b"|")
    keys[h.hexdigest()].append(u["bird"] + "/" + u["file"])
dups = [v for v in keys.values() if len(v) > 1]
print(f"single units with identical spike trains to all call stimuli: {len(dups)} groups")
for v in dups[:10]:
    print("  ", v)

Clock. In 50 ms windows around onset, a Kruskal-Wallis test of the spike counts across the nine call types, per
single unit. Before onset the counts cannot depend on the call that follows; after onset they can.

In [ ]:
# Contrast between conditions around the trial start: in 50 ms windows from -300 to
# 600 ms, a Kruskal-Wallis test of the trial spike counts across the nine call types,
# per unit. Before onset the counts cannot depend on the call type.
wins = np.arange(-0.3, 0.6, 0.05)
frac = []
rate = []
for w0 in wins:
    sig, r = 0, []
    for u in S:
        st = elie.call_stims(u)
        grp = collections.defaultdict(list)
        for s in st:
            for t in s["trials"]:
                n = int(((t >= w0) & (t < w0 + 0.05)).sum())
                grp[s["ct"]].append(n); r.append(n)
        vals = list(grp.values())
        try:
            p = stats.kruskal(*vals).pvalue
        except ValueError:
            p = 1.0
        sig += p < 0.01
    frac.append(sig / len(S)); rate.append(np.mean(r) / 0.05)
clock = pd.DataFrame({"window_start_ms": (wins * 1000).round().astype(int), "fraction_units_p<0.01": np.round(frac, 3),
                      "mean_rate_spikes_per_s": np.round(rate, 2)})
print(clock.to_string(index=False))
pre, post = clock[clock.window_start_ms < 0], clock[clock.window_start_ms >= 0]
print(f"before onset: {100 * pre.iloc[:, 1].min():.1f}% to {100 * pre.iloc[:, 1].max():.1f}% of units at p < 0.01, "
      f"rate {pre.iloc[:, 2].min():.2f} to {pre.iloc[:, 2].max():.2f} spikes/s; from onset: "
      f"{100 * post.iloc[:, 1].min():.1f}% to {100 * post.iloc[:, 1].max():.1f}%, peak rate {post.iloc[:, 2].max():.2f} spikes/s")

Before onset 0.5% to 1.3% of units differ across call types at p < 0.01, near the 1% false-positive rate, and
the mean rate is flat at 6.33 to 6.41 spikes/s; from the 0 to 50 ms window on, 42.2% to 65.3% do and the rate
rises to 23.30 spikes/s in the 50 to 100 ms window. The spike times share the clock of the stimulus onsets, for
the population; a check per unit was not made.


## units-404

> The analysis uses 404 of the 914 isolated single units, those from which call type could be decoded above
> chance in the authors' earlier study.

Criterion: reading (a), a list of the 404 units from the data set's documentation or the 2015 paper; reading (b),
the 2015 decoding selection redone as that paper describes it. Blocked as methods_underspecified when neither
can be carried out.

In [ ]:
n_single = len(S)
n_aud = int((sd.iar >= 2.3).sum())
print(f"files labelled single units (sortType single or singl): {n_single}")
print(f"  of which I_AR >= 2.3 bits/s (the 2015 threshold for auditory spike-sorted units): {n_aud}")
print(f"spike-sorted files (_ss) of any sortType: {int(df.spike_sorted_name.sum())}")
print("expected: 914 single auditory units, of which 404 selected by the 2015 decoding")
print("per bird, single / single with I_AR >= 2.3:")
print(sd.assign(aud=sd.iar >= 2.3).groupby("bird").agg(single=("file", "size"), auditory=("aud", "sum")).to_string())
attrs = sorted({k for u in U for k in u if k not in ("stims", "class_info", "groups", "attr_names")})
print("unit-level fields read from every file:", attrs)
print("groups found in files:", sorted({g for u in U for g in u["groups"]}))
ss = df[df.spike_sorted_name]
print(f"\nspike-sorted files (_ss) of any sortType with I_AR >= 2.3 bits/s: {int((ss.iar >= 2.3).sum())} "
      f"(description: 1401 spike-sorted auditory units)")
print(pd.crosstab(ss.sortType.astype(str), ss.iar >= 2.3).rename(columns={True: "I_AR>=2.3", False: "I_AR<2.3"}).to_string())


The data description gives the snippet SNR that separates the 914 single auditory units from the 1401 auditory spike-sorted ones: "the difference between the maximum and the minimum of the average snippet" over "the standard deviation of this measure across all snippets", with a threshold of 5 (also the paper, Methods). The snippet files and their layout (18 float32 samples per spike, indexed by `spike_ids`) are read as `CRCNSCode/h5_plot_one_spike.m` in the archive reads them. Below: the snippet files per bird and site, and the SNR of each spike-sorted unit on its call protocol, under the two readings of "this measure" (`elie` module docstring).

In [ ]:
f32 = sorted({(u["bird"], n, k) for u in U for n, k in u["f32_files"].items()})
f32t = pd.DataFrame([{"bird": b, "file": n, "site": elie.f32_key(n)[0], "protocol": elie.f32_key(n)[1], "snippets": k}
                     for b, n, k in f32])
print(f"snippet files: {len(f32t)}")
print(f32t.to_string(index=False))
print("\nsites with h5 files and no call-protocol snippet file:")
have = {(r.bird, r.site) for r in f32t.itertuples() if r.protocol.startswith("Call")}
print(sorted({(u["bird"], u.get("site")) for u in U} - have))
ss = df[df.spike_sorted_name].copy()
for k in ("snr_a", "snr_b", "snr_n", "snr_n_files"):
    ss[k] = [u.get(k, np.nan) for u in U if "_ss" in u["file"]]
ss["aud"] = ss.iar >= 2.3
print(f"\nspike-sorted units with an SNR: {ss.snr_a.notna().sum()} of {len(ss)}; "
      f"computed from two candidate files: {(ss.snr_n_files > 1).sum()}")
for col in ("snr_a", "snr_b"):
    print(f"\nSNR reading {col[-1].upper()}: quantiles (5, 25, 50, 75, 95%) by sortType")
    print(ss.groupby(ss.sortType.astype(str))[col].quantile([.05, .25, .5, .75, .95]).unstack().round(2).to_string())
    a = ss[ss.aud]
    print(f"auditory spike-sorted units (I_AR >= 2.3) with SNR > 5: {(a[col] > 5).sum()} of {len(a)} "
          f"(description: 914 of 1401); among those labelled single: {(a[col] > 5)[a.single].sum()} of {a.single.sum()}")
    print("  by bird:", a.groupby("bird")[col].apply(lambda x: int((x > 5).sum())).to_dict())
    for th in (3, 4, 5, 6):
        print(f"  threshold {th}: {(a[col] > th).sum()} auditory spike-sorted units")
print("\nsortType single against SNR > 5 (reading A), auditory spike-sorted units:")
print(pd.crosstab(ss[ss.aud].sortType.astype(str), ss[ss.aud].snr_a > 5).to_string())


What the h5 files hold beyond the spikes: the dataset paths of `class_info` and `extra_info` over all files, and how many files hold each, then the names of the attributes of every group and dataset in the files (root, protocol, stimulus and trial groups, and everything under `class_info` and `extra_info`), with how many files hold each. An anatomical area or a coordinate of the site would be in one of these.


In [ ]:
import re as _re
for key in ("class_info_paths", "extra_info_paths"):
    c = collections.Counter(_re.sub(r"\(\d+, \d+\)", "(shape)", _re.sub(r"^[A-Za-z0-9]+/", "<protocol>/", p))
                            for u in U for p in u[key])
    print(f"{key[:-6]}: {sum(bool(u[key]) for u in U)} files hold it; dataset paths (protocol and shapes generalised):")
    for p, n in sorted(c.items()):
        print(f"   {n:5d}  {p}")
print("example, one file:", U[0]["file"])
for p in U[0]["extra_info_paths"]:
    print("   extra_info/" + p)
c = collections.Counter(a for u in U for a in u["attr_names"])
print("attribute names over all files (path generalised, then name), and the number of files holding each:")
for a, n in sorted(c.items()):
    print(f"   {n:5d}  {a}")


The files hold no list or flag of the 404 units, and the data description and the 2015 paper (Elie &
Theunissen 2015, Eur J Neurosci 41:546-567, read at PMC4347984) give none, so reading (a) cannot be carried out.

Reading (b) starts from the 914 single auditory units. The I_AR reading matches the authors': 1401 spike-sorted
files of any `sortType` have a stored I_AR of at least 2.3 bits/s, the 1401 auditory spike-sorted units of the
description. The description then keeps the units with a snippet SNR above 5. Every site has a snippet file for
its call protocol, and 1452 of the 1453 spike-sorted units get an SNR, but the threshold leaves 461 auditory units
(reading A of the SNR) or 274 (reading B), not 914; at a threshold of 4, reading A leaves 772, and at 3, 1120. So
the SNR as described, computed on the call-protocol snippets, does not recover the 914, and the `sortType` label
(1246 auditory units labelled single) does not either. The 2015 decoding would also have to be run on a set that
is not the paper's.

The extra datasets of each file are coherence (`class_info`) and inter-spike-interval and section statistics
(`extra_info`); none holds an anatomical area or a coordinate. The attributes of every group and dataset
(root, protocol, stimulus and trial groups, and everything under `class_info` and `extra_info`, which carry none)
give the site, electrode and depths (`ldepth`, `rdepth`) and the stimulus and response statistics, but no area or
coordinate either.

**Result: blocked.** The data set holds 1281 files labelled single units, 1246 of them with I_AR of at least 2.3
bits/s, and the snippet SNR leaves 461 or 274; neither the 914 nor the 404 can be identified.


## Cost of the paper's pipeline

Every claim from units-337 on needs the locally adaptive KDE rate of each unit and stimulus, and its 10
jackknife estimates (Methods R), before the Monte Carlo cumulative information. The time of the KDE alone, on a
sample of call stimuli of three auditory single units:

In [ ]:
# Cost of the paper's rate estimate (Methods R) on the data: time the locally
# adaptive KDE on the call stimuli of a few auditory single units, for the full
# set of trials and one jackknife set, and scale to 404 units x 11 estimates per stimulus.
rng_cost = np.random.default_rng(404)
aud = [u for u in S if elie.iar(u) >= 2.3]
sample = [aud[i] for i in rng_cost.choice(len(aud), 3, replace=False)]
n_kde, t_kde = 0, 0.0
for u in sample:
    for s in elie.call_stims(u)[:10]:
        tr = [t[(t >= 0) & (t < 0.6)] * 1000 for t in s["trials"]]
        for drop in (None, 0):
            use = [t for i, t in enumerate(tr) if i != drop]
            t1 = time.time()
            elie.kde_rate(np.concatenate(use), len(use), 600)
            t_kde += time.time() - t1; n_kde += 1
per = t_kde / n_kde
mean_stims = sd[sd.iar >= 2.3].n_call_stims.mean()
est_h = per * 404 * mean_stims * 11 / 3600
print(f"{n_kde} KDE estimates on 3 units: {per:.2f} s each")
print(f"estimate for the rates alone (404 units x {mean_stims:.0f} stimuli x 11 estimates): {est_h:.0f} h on one core, "
      f"{est_h / 4:.0f} h on 4 cores")

The rate estimates alone would take more than 100 hours on one core, and about 30 hours on
the runner's 4 cores, before any Monte Carlo estimate of cumulative information (the model neuron's 40 bins, with 4
stimuli, take about 5 minutes below; a unit has 60 bins and about 90 stimuli). This is beyond the limit of a
notebook even if the 404 units could be identified, so every claim from units-337 on, and fig2a-poisson-test, which
needs the same rates, would also be blocked as `too_large`. The time per estimate varies between runs; the printed
estimate is the one for this run.


## Variants on all auditory single units (not deciding any outcome)

The spike-count measures of s2-spike-counts, fig2cd-fano-factor and fig2b-noise-correlations, and the coherence
part of s1-temporal-resolution, on all 1246 single units with I_AR of at least 2.3 bits/s. The claims rest on the
404 units; these numbers describe the data set's units and decide nothing.

In [ ]:
# Not deciding any outcome: the spike-count measures of s2-spike-counts,
# fig2cd-fano-factor, fig2b-noise-correlations and the coherence part of
# s1-temporal-resolution, on all auditory single units (I_AR >= 2.3 bits/s),
# since the 404 units cannot be identified.
rows = []
for u in aud:
    cs = elie.counts(u)
    allc = np.concatenate([c.ravel() for c in cs])
    sc1, sc2 = elie.stim_corr(cs)
    rows.append({"bird": u["bird"], "max_count": int(allc.max()), "n8": int((allc == 8).sum()),
                 "n_bins": allc.size, "sum": int(allc.sum()),
                 "mean_trialavg": float(np.mean([c.mean(0).mean() for c in cs])),
                 "fano_a": elie.fano(cs, True), "fano_b": elie.fano(cs, False),
                 "noise_corr": elie.noise_corr(cs), "stim_corr_over_stims": sc1, "stim_corr_within_stim": sc2,
                 "coh_frac_50": elie.coherence_fraction(u)})
V = pd.DataFrame(rows)
print(f"units: {len(V)}")
print(f"s2: largest count in a 10 ms bin {V.max_count.max()}; bins with 8 spikes {V.n8.sum()}; "
      f"bins with more than 8: {(V.max_count > 8).sum()} units have one; mean count per bin over trials "
      f"{V['sum'].sum() / V.n_bins.sum():.3f}, over trial-averaged counts {V.mean_trialavg.mean():.3f} "
      f"(paper, 404 units: 8, 1, 0.108)")
print("  distribution of the largest count per unit:", dict(sorted(collections.Counter(V.max_count).items())))
for col in ["fano_a", "fano_b"]:
    x = V[col].dropna()
    print(f"Fano ({col}): mean {x.mean():.3f}, SEM {x.std() / np.sqrt(len(x)):.3f}, range {x.min():.2f} to {x.max():.2f}; "
          f"t-test vs 1 p = {stats.ttest_1samp(x, 1).pvalue:.2g}; Wilcoxon vs 1 p = {stats.wilcoxon(x - 1).pvalue:.2g} "
          f"(paper: 1.04, SEM 0.04)")
x = V.noise_corr.dropna()
print(f"noise correlation: median {x.median():.3f}, mean {x.mean():.3f}, skewness {stats.skew(x):.2f}, range {x.min():.2f} to {x.max():.2f}")
for col in ["stim_corr_over_stims", "stim_corr_within_stim"]:
    x = V[col].dropna()
    print(f"stimulus correlation ({col}): median {x.median():.3f}, mean {x.mean():.3f}, range {x.min():.2f} to {x.max():.2f} (paper: around 0.4)")
x = V.coh_frac_50 * 100
print(f"coherence information below 50 Hz: {x.mean():.1f}% +- {x.std():.1f}% (mean +- SD); units losing more than 5%: "
      f"{(x < 95).sum()} of {len(x)} (paper, 404 units: 96.7 +- 6.9%, 88 of 404)")
print("\nby bird (medians):")
print(V.groupby("bird")[["fano_a", "noise_corr", "stim_corr_over_stims", "coh_frac_50"]].median().round(3).to_string())

On the 1246 auditory single units, call protocol only: the largest count in a 10 ms bin is 9 (4 units) with 442
bins at 8, and the mean count per bin is 0.136 (0.126 over trial-averaged counts), against 8, 1 and 0.108 for the
paper's 404. The time-averaged Fano factor is 1.028 (SEM 0.002) under the code's reading and 1.054 (SEM 0.003)
leaving out empty bins, both significantly above 1 at this n. The noise correlation has median 0.151 and positive
skew (1.07); the stimulus correlation has median 0.291 over stimuli and 0.243 within stimuli. The stored coherence
puts 65.8% +- 19.0% of I_AR below 50 Hz. The stored curve runs from 4 to 496 Hz, and nothing in the files or the
description shows that it is the computation of S1 Fig, so this number says nothing about the paper's 96.7%. The
power part of s1-temporal-resolution needs the KDE rates and was not computed (see the cost above).


Check of the Monte Carlo estimate against exact enumeration, on a synthetic case small enough to enumerate (5 stimuli, 3 bins, categories {0, 0, 0, 1, 2}): for stimuli and for categories, with categories weighted equally as in `inst_info` and the paper's code (`info_model_Calculus_wrapper2.m`). Bin 1 of the cumulative information must also equal the instantaneous information.

In [ ]:
mu_syn = np.array([[0.2, 1.5, 0.3], [0.2, 0.2, 1.0], [0.5, 0.1, 0.1], [1.2, 0.4, 2.0], [0.05, 0.9, 0.6]])
g_syn = np.array([0, 0, 0, 1, 2])
for lab, g in (("stimuli", None), ("categories", g_syn)):
    ex, mc = elie.cum_info_exact(mu_syn, g), elie.cum_info_mc(mu_syn, 200_000, 11, g)
    print(f"{lab}: exact {ex.round(4)}, Monte Carlo {mc.round(4)}, largest |difference| {np.abs(ex - mc).max():.4f} bits; "
          f"instantaneous at bin 1 {elie.inst_info(mu_syn, g)[0]:.4f}")


## fig1-model-neurons

> For simulated Poisson neurons (rate, onset and temporal codes, 4 stimuli, 10 trials), the KDE recovers the
> model rates with an average error below 0.02 spikes/ms, the cumulative information approaches the 2-bit
> maximum, and the temporal neuron's peak instantaneous information is 0.5 bits.

Criterion: with the tutorial's model cells, mean absolute difference between KDE and model rates below 0.02
spikes/ms, the cumulative information of each model neuron reaching at least 1.8 bits before it ends, and the
temporal neuron's peak instantaneous information within 0.5 +- 0.05 bits.

The repository holds one model cell, a temporal neuron (each stimulus drives a response at a different time).
The rate and onset neurons are not there. The KDE is the Python port of ssvkernel (`adaptivekde`).

The first cell uses a simplified procedure: the jackknife bias correction N I - (N-1) mean(I_k) with jackknife set
k used for every stimulus, and one set of 200,000 Monte Carlo samples for all bins, drawn from the paper's
proposal (the product of the per-bin marginals; Methods, The cumulative information). The second cell runs the
paper's procedure as its code does it (Methods, The cumulative information, and Bias correction;
`cumulative_info_poisson_model_MCJK_wrapper.m`, `cumulative_info_poisson_model_calculus_MCJK.m` and the tutorial
at the pinned commit): a new estimate for each end bin, chunks of 100,000 samples until the error is below 0.2 bits
or 5,000,000 samples, NB = 20 bootstraps of the jackknife (the jackknife sets paired across stimuli at random),
and no further bins once the error exceeds 0.6 bits. In the code the jackknife sets are evaluated on the samples
drawn for the full data, so the replicates share their random numbers; this is the code's procedure, not an
assumption. The second cell decides the outcome.


In [ ]:
import io, fsspec, scipy.io as sio
REPO = "https://raw.githubusercontent.com/julieelie/PoissonTimeVaryingInfo/978020328321c7ed83be12c0152bed85ec77e218/"
with fsspec.open(REPO + "ModelCell_InfoCumInfo_Tutorial.mat", block_size=2**24).open() as f:
    M = sio.loadmat(io.BytesIO(f.read()))
n_stim, n_tr, dur = int(M["NStims"][0, 0]), int(M["NTrials"][0, 0]), int(M["NR_Duration"][0, 0])
model_bins = M["Input_Spike_count"]            # model mean count per 10 ms bin, stimulus x bin
print(f"tutorial model cell: {n_stim} stimuli, {n_tr} trials, {dur} ms at {int(M['Response_samprate'][0, 0])} Hz, "
      f"{model_bins.shape[1]} bins of {int(M['Step'][0, 0])} ms")
print("model mean count per 10 ms bin, range", model_bins.min(), "to", model_bins.max())
print("start (ms) of the bins where each stimulus drives a response (model count > 0.1):",
      [[int(b) for b in np.flatnonzero(r > 0.1) * 10] for r in model_bins])
arr = [M["Spike_array"][0, 0][f"stim{s + 1}"] for s in range(n_stim)]   # trial x 1 ms
spk = [[np.repeat(np.flatnonzero(a[t]), a[t][a[t] > 0].astype(int)) + 1.0 for t in range(n_tr)] for a in arr]
print("spikes per stimulus (all trials):", [int(a.sum()) for a in arr])
print("spikes per trial, model (sum of the mean counts per bin) against observed:",
      np.round(model_bins.sum(1), 2), np.round([a.sum() / n_tr for a in arr], 2))

rate = np.array([elie.kde_rate(np.concatenate(sp), n_tr, dur) for sp in spk])           # spikes/ms
jk = np.array([[elie.kde_rate(np.concatenate([x for i, x in enumerate(sp) if i != k]), n_tr - 1, dur)
                for sp in spk] for k in range(n_tr)])                                     # set x stim x ms
saved = np.array([M["Spike_filtered"][0, 0][f"stim{s + 1}"].ravel() for s in range(n_stim)])
print(f"KDE here against the KDE saved by the MATLAB tutorial: mean |diff| {np.abs(rate - saved).mean():.5f}, "
      f"max {np.abs(rate - saved).max():.5f} spikes/ms")
model = np.repeat(model_bins / 10.0, 10, axis=1)                                          # spikes/ms at 1 kHz
mae = np.abs(rate - model).mean(1)
print("mean |KDE - model rate| per stimulus (spikes/ms):", np.round(mae, 4), "; all:", round(float(mae.mean()), 4),
      "; RMS:", round(float(np.sqrt(((rate - model) ** 2).mean())), 4))

mu = np.array([elie.bin_means(r) for r in rate])
mu_jk = np.array([[elie.bin_means(r) for r in s] for s in jk])
mu_model = np.maximum(model_bins, elie.MU_MIN)
N = n_tr

def jackknife(full, reps):
    """Bias-corrected value N*I - (N-1)*mean(I_k) and its jackknife standard error."""
    reps = np.asarray(reps)
    bc = N * full - (N - 1) * reps.mean(0)
    se = np.sqrt((N - 1) / N * ((reps - reps.mean(0)) ** 2).sum(0))
    return bc, se

ii, ii_se = jackknife(elie.inst_info(mu), [elie.inst_info(m) for m in mu_jk])
ii_model = elie.inst_info(mu_model)
SEED_F1 = 1
NS = 200_000
ci_full = elie.cum_info_mc(mu, NS, SEED_F1)
ci, ci_se = jackknife(ci_full, [elie.cum_info_mc(m, NS, SEED_F1) for m in mu_jk])
ci_model = elie.cum_info_mc(mu_model, NS, SEED_F1)
tab = pd.DataFrame({"bin_end_ms": np.arange(1, mu.shape[1] + 1) * 10, "inst_bc": ii.round(3), "inst_se": ii_se.round(3),
                    "inst_model": ii_model.round(3), "cum_bc": ci.round(3), "cum_se": ci_se.round(3),
                    "cum_model": ci_model.round(3)})
print(tab.to_string(index=False))
print(f"peak bias-corrected instantaneous information {ii.max():.3f} bits at bin ending {10 * (ii.argmax() + 1)} ms "
      f"(model rate: {ii_model.max():.3f}); uncorrected peak {elie.inst_info(mu).max():.3f}")
print(f"cumulative information: largest {ci.max():.3f} bits, last bin {ci[-1]:.3f} bits (model rate: largest {ci_model.max():.3f}, "
      f"last bin {ci_model[-1]:.3f}); "
      f"largest jackknife SE {ci_se.max():.3f} bits; maximum log2(4) = 2 bits")
print("same with the MATLAB tutorial's saved results: peak instantaneous",
      round(float(M["Info_bcorr"].max()), 3), "; last cumulative", round(float(M["Icum_EstMonteCarloOpt_bcorr"][0, -1]), 3))
# Variant, not the code's procedure: jackknife replicates with independent Monte Carlo draws (seeds 2 to 11).
ci_ind, _ = jackknife(ci_full, [elie.cum_info_mc(m, NS, 2 + k) for k, m in enumerate(mu_jk)])
print(f"variant, independent draws per replicate: largest {ci_ind.max():.3f} bits, last bin {ci_ind[-1]:.3f} bits")

In [ ]:
# The paper's procedure (Methods, The cumulative information; Bias correction; code
# cumulative_info_poisson_model_MCJK_wrapper.m and _calculus_MCJK.m): a new Monte Carlo
# estimate for each end bin, chunks of 100,000 samples until the error is below 0.2 bits or
# 5,000,000 samples, NB = 20 bootstraps of the jackknife (random pairing of the jackknife
# sets across stimuli), the jackknife sets evaluated on the samples drawn for the full data,
# and no further bins once the error exceeds 0.6 bits.
t1 = time.time()
NB = 20
ii_p, ii_p_err = elie.inst_info_bootjk(mu, mu_jk, N, NB, seed=3)
r = elie.cum_info_mcjk(mu, mu_jk, N, NB, seed=4)
r["bc"][0] = ii_p[0]   # bin 1 is the instantaneous value, as in the tutorial
print(f"run time {time.time() - t1:.0f} s; bins computed: 1 to {r['stopped_at']} of {mu.shape[1]}")
tab_p = pd.DataFrame({"bin_end_ms": np.arange(1, mu.shape[1] + 1) * 10, "inst_bc": ii_p.round(3), "inst_err": ii_p_err.round(3),
                      "cum_bc": r["bc"].round(3), "cum_raw": r["raw"].round(3), "cum_err": r["err"].round(3),
                      "mc_samples": r["n_samples"], "matlab_cum_bc": M["Icum_EstMonteCarloOpt_bcorr"].ravel().round(3),
                      "matlab_err": M["Icum_EstMonteCarloOpt_err"].ravel().round(3), "matlab_samples": M["MC_Samp"].ravel()})
print(tab_p.to_string(index=False))
print(f"paper's procedure: peak bias-corrected instantaneous information {ii_p.max():.3f} bits at bin ending "
      f"{10 * (ii_p.argmax() + 1)} ms (error {ii_p_err[ii_p.argmax()]:.3f})")
print(f"paper's procedure: cumulative information largest {np.nanmax(r['bc']):.3f} bits, last bin {r['bc'][r['stopped_at'] - 1]:.3f} bits; "
      f"bins at or above 1.8 bits: {int(np.nansum(r['bc'] >= 1.8))}; largest error {np.nanmax(r['err']):.3f} bits; "
      f"bins that reached 5,000,000 samples: {int(np.nansum(r['n_samples'] >= 5_000_000))}")
print(f"MATLAB tutorial, same procedure: largest {np.nanmax(M['Icum_EstMonteCarloOpt_bcorr']):.3f}, last "
      f"{M['Icum_EstMonteCarloOpt_bcorr'][0, -1]:.3f}, largest error {np.nanmax(M['Icum_EstMonteCarloOpt_err']):.3f} bits")


**Result: partially reproduced.** On the tutorial's temporal neuron the KDE differs from the model rate by 0.009
spikes/ms on average (below 0.02). Under the paper's procedure the bias-corrected cumulative information reaches
1.982 bits (at least 1.8; 14 bins at or above 1.8, last bin 1.884), all 40 bins are computed with a largest error of
0.234 bits, and the peak bias-corrected instantaneous information is 0.520 bits (within 0.5 +- 0.05). The MATLAB
tutorial's saved results with the same procedure are close (largest 1.981, last 1.894). The simplified procedure
gives the same conclusion (1.928 bits; peak 0.519 bits), and so does the model rate (cumulative largest 1.990, last bin
1.956; peak 0.540).
With independent Monte Carlo draws per jackknife replicate, which is not the code's procedure, the cumulative
information rises to 2.206 bits, above the 2-bit maximum, and ends at 0.803. The Monte Carlo estimate agrees with
exact enumeration to 0.0034 bits for stimuli and 0.0019 bits for categories on a synthetic case. The rate and onset
neurons of Fig 1 are not in the repository, so the claim holds for one of its three model neurons.


## fig3-5-example-neurons

The paper does not name the three example units, and the criterion blocks the claim when they cannot be identified. The S4 to S6 Fig raster images are not among the materials of this attempt (the paper was supplied as text, with the legends only), so they were not inspected and no match against the data was attempted.

## Summary

| claim | outcome |
|---|---|
| units-404 | blocked (methods_underspecified): 914 and 404 units not identifiable; 1281 files labelled single, snippet SNR > 5 leaves 461 or 274 |
| units-337, fig2a-poisson-test, fig2cd-fano-factor, fig2b-noise-correlations, s1-temporal-resolution, s2-spike-counts | blocked (methods_underspecified, from units-404) |
| fig6a-onset-sustained-information, fig6c-new-information, fig6b-fixed-rate-fraction, fig7a-time-constants, fig7b-k300, k300-rate-regression, fig9a-mean-cii, fig9a-top-quartile, s7-invariant-information, fig9b-cii-tau-ratio, fig9c-cii-k300, cii-unimodal | blocked (methods_underspecified, from units-337; also too_large) |
| fig3-5-example-neurons | blocked (methods_underspecified): the three units are not named |
| table1-anatomy-counts, fig10-cii-distance, fig10-cii-area-anova, fig10-tau-k300-distance, fig10-no-anatomy-categorical | blocked (data_missing): no anatomical location in the data set; also rest on units-337 |
| fig1-model-neurons | partially reproduced, on the one model neuron in the paper's repository |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

Differences from the paper's method: the KDE is a Python port of ssvkernel; the model neuron is the only one
published. The cumulative information of fig1 follows the paper's code (adaptive sample size, bootstrap of the
jackknife, 0.6-bit stop); the simplified estimate is shown beside it.

What was found about the data set: the files do not mark the 914 single auditory units or the 404 analysed units
(1281 files are labelled single units, 1246 pass the 2.3 bits/s auditory threshold, and the snippet SNR described
as the single-unit criterion leaves 461 or 274, not 914); the 1401 auditory spike-sorted units of the description
are found; the layout of the snippet files is documented only by a plotting function in the archive's code; the
anatomical locations and coordinates of the recording sites are not in the data set (depths are in the file
names); the number of stimuli per unit in the call protocol matches the paper's mean and minimum but not its
spread or maximum, and the description's 144 matches no count; up to 12 trials per stimulus, against a stated
maximum of 11; three files store `sortType` as "singl"; the description names one bird WhiBlu5896M where the
files say WhiBlu5396M.
